In [17]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *
from locallib.box import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [19]:
customer_name = 'ITALGAS'
file_name = f"{customer_name}_Yearly_Analysis.xlsx"

In [20]:
# FIXED: Proper parentheses, correct SQL, execute call outside Query call, and fetch result
query_string = f"""
    SELECT * FROM KPI_OutputExcelLocation 
    WHERE CustomerId = (
        SELECT CustomerId FROM KPI_Customer WHERE name = '{customer_name}'
    )
"""
result = Query(query=query_string).execute(KPIHub_Conn)
if result.empty:
    raise ValueError(f"No output Excel location found for customer: {customer_name}")
box_folder_id = result['BoxFolderId'].values[0]

In [21]:
#Get the columns of the view and get the KPI definitions
cols = Query("PRAGMA table_info('Yearly_KPI')").execute(KPIHub_Conn)
# Remove 'customerUtiliyation' and 'StandarTuiliyation' from cols DataFrame if present
cols = cols[~cols['name'].isin(['CustomerUtilization', 'StarndardUtilization','DaysCount','SurveysCarDay','PeakAboveSATCount','CumulativeAssetCoveredLengthKm','TargetDurationHours'])]
cols.db.set_query("SELECT * FROM KPI_Definition WHERE name IN (SELECT name from temp_KPI)")
kpi_col = cols.db.execute(KPIHub_Conn, source_col = 'name', temp_table_name = 'temp_KPI')
output_dict = {kpi_col['Name']: [kpi_col['Unit'], kpi_col['Description'], kpi_col['Formula']] for _, kpi_col in kpi_col.iterrows()}

In [22]:
kpi_data = Query(f"SELECT {', '.join(cols['name'])} FROM Yearly_KPI WHERE CustomerName = '{customer_name}' AND BoundaryRegion IS NOT NULL").execute(KPIHub_Conn)
years= Query(f"SELECT DISTINCT Year FROM Yearly_KPI WHERE CustomerName = '{customer_name}' AND BoundaryRegion IS NOT NULL").execute(KPIHub_Conn)

In [23]:
with pd.ExcelWriter(file_name) as writer:
    for _, region_row in years.iterrows():
        year = region_row['Year']
        # Select the export_df according to region (handle null safely)

        export_df = kpi_data[kpi_data['Year'] == year].copy()
        sheet_name = f"KPI {year} Yearly"

        # Fill PeakAboveSATCount with 0, but only if column exists
        if 'PeakAboveSATCount' in export_df.columns:
            export_df['PeakAboveSATCount'] = export_df['PeakAboveSATCount'].fillna(0)

        # Process the column name
        units = []
        for col in export_df.columns:
            if col in kpi_col['Name'].values:
                units.append(kpi_col.loc[kpi_col['Name'] == col, 'Unit'].values[0])
            else:
                units.append("")

        # Write the filtered DataFrame to the first sheet
        # Write columns and units as first two rows, then export the rest of the DataFrame
        rows = export_df.values.tolist()
        full_rows = [export_df.columns.tolist(), units] + rows
        temp_df = pd.DataFrame(full_rows)

        temp_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

        # Post-process the sheet for bold and center alignment of the first two columns
        worksheet = writer.sheets[sheet_name]
        # Create bold and center formats
        bold_center = writer.book.add_format({'bold': True, 'align': 'center'})
        center = writer.book.add_format({'align': 'center'})

        # The first two rows (headers and units): apply bold and center format to *all* columns, not just columns 0 and 1
        for row_idx in range(2):
            for col_idx in range(len(full_rows[row_idx])):
                worksheet.write(row_idx, col_idx, full_rows[row_idx][col_idx], bold_center)  # overwrite with bold+center

        # All other rows: just center the first two columns
        for i, row in enumerate(rows, start=2):
            for col in range(2):
                worksheet.write(i, col, row[col], center)
        # Add a colored line (cell border) at the bottom of all cells in the second row (units row)
        bottom_border_format = writer.book.add_format({'bottom': 1, 'bottom_color': '#000000', 'align': 'center', 'bold': True})
        for col_idx in range(len(full_rows[1])):
            worksheet.write(1, col_idx, full_rows[1][col_idx], bottom_border_format)

        # Auto-adjust column widths based on the maximum length in each column
        for idx, col in enumerate(export_df.columns):
            # Find the max length of any value in this column (including header and units)
            max_len = max(
                [len(str(col)), len(str(units[idx]))] +
                [len(str(row[idx])) for row in rows]
            )
            # Set the width (add some padding)
            worksheet.set_column(idx, idx, max_len + 2)

    # Write the key and its list of [Unit, Description, Formula Used] as columns in the second sheet
    desc_rows = []
    for key, value in output_dict.items():
        if isinstance(value, list) and len(value) == 3:
            # Value is a list: [Unit, Description, Formula Used]
            row = [key] + value
        else:
            # Fallback in case the dictionary isn't formatted as expected
            row = [key, "", "", ""]
        desc_rows.append(row)
    columns = ["Key", "Unit", "Description", "Formula Used"]
    desc_df = pd.DataFrame(desc_rows, columns=columns)
    desc_df.to_excel(writer, sheet_name="KPI Descriptions", index=False)


In [24]:
box_obj = BoxFile(local_path = file_name, box_folder_id = box_folder_id)
box_obj.upload()
box_obj.delete()

Request "GET https://api.box.com/2.0/folders/401896871043" failed with ConnectionError exception: ConnectionError(ProtocolError('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')))


Request "POST https://upload.box.com/api/2.0/files/2360331323746/content" failed with ConnectionError exception: ConnectionError(ProtocolError('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')))
